# 🏋️ Training Demo

End-to-end training workflow:
1. Generate simulated data
2. Train Sklearn → DNN → Hybrid models
3. Compare results

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import yaml
import matplotlib.pyplot as plt

from src.pipeline.data_generator import DataGenerator
from src.pipeline.data_loader import DataLoaderFactory
from src.models.classifier import SklearnClassifier, DNNClassifier, HybridClassifier
from src.training.trainer import Trainer
from src.evaluation.evaluator import ModelEvaluator

%matplotlib inline

In [ ]:
with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

# Use smaller dataset for notebook demo
DEMO_SAMPLES = 10_000
np.random.seed(config['project']['seed'])

## 1. Generate Data

In [ ]:
gen = DataGenerator(config)
X, y, accent_ids, noise_ids = gen.generate_fast(num_samples=DEMO_SAMPLES)
gen.save(X, y, accent_ids, noise_ids)

print(f'X shape: {X.shape}')
print(f'Classes: {np.unique(y)}')

## 2. Load Splits

In [ ]:
loader = DataLoaderFactory(config)
X_train, y_train, _, _ = loader.load_arrays('train')
X_val, y_val, _, _ = loader.load_arrays('val')
X_test, y_test, _, _ = loader.load_arrays('test')

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 3a. Train Sklearn Model

In [ ]:
sk_model = SklearnClassifier(config)
trainer = Trainer(config)
trainer.train_sklearn(sk_model, X_train, y_train, X_val, y_val)

## 3b. Train DNN Model

In [ ]:
feature_dim = X_train.shape[1]
num_classes = config['data']['num_classes']

dnn_model = DNNClassifier(config, feature_dim, num_classes)
trainer_dnn = Trainer(config)
trainer_dnn.train_dnn(dnn_model, X_train, y_train, X_val, y_val)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(trainer_dnn.history['train_loss'], label='Train Loss')
ax1.plot(trainer_dnn.history['val_loss'], label='Val Loss')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(trainer_dnn.history['val_acc'], label='Val Accuracy', color='green')
ax2.set_title('Validation Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

## 3c. Train Hybrid Model

In [ ]:
hybrid_model = HybridClassifier(config, feature_dim, num_classes)
trainer_hyb = Trainer(config)
trainer_hyb.train_hybrid(hybrid_model, X_train, y_train, X_val, y_val)

## 4. Evaluate All Models

In [ ]:
evaluator = ModelEvaluator(config)

print('\n=== Sklearn ===')
sk_report = evaluator.evaluate(sk_model, X_test, y_test)

print('\n=== DNN ===')
dnn_report = evaluator.evaluate(dnn_model, X_test, y_test)

print('\n=== Hybrid ===')
hybrid_report = evaluator.evaluate(hybrid_model, X_test, y_test)

## 5. Model Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Sklearn': {k: v for k, v in sk_report.items() if isinstance(v, float)},
    'DNN': {k: v for k, v in dnn_report.items() if isinstance(v, float)},
    'Hybrid': {k: v for k, v in hybrid_report.items() if isinstance(v, float)},
})

display(comparison.style.highlight_max(axis=1, color='lightgreen'))

comparison.plot(kind='bar', figsize=(10, 5), colormap='Set2')
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()